# Fase 2.1 — EDA descriptivo: población, distribuciones y faltantes

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

Abro el análisis exploratorio describiendo **quiénes son** los créditos de libranza, sin mirar todavía su relación con el incumplimiento. Esa relación va en el notebook 06 y solo con cohortes anteriores a 2025.

Las preguntas de este notebook son:

1. ¿Cómo creció y cambió la cartera en el tiempo (volumen, modalidad de recaudo, clientes recurrentes, refinanciaciones)?
2. ¿Qué tan selectiva es la aprobación? Esto define el tamaño del sesgo de selección (*reject inference*).
3. ¿Qué tan concentrada está la cartera en pocos empleadores?
4. ¿Cómo se distribuyen las variables candidatas y dónde hay valores atípicos?
5. ¿Qué falta, cuándo falta y por qué? Distingo el faltante **estructural** (la fuente no existía) del faltante **ocasional**.
6. ¿Qué variables son redundantes entre sí?

Población: créditos de libranza no anulados (1.087). Todas las salidas son agregadas.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from tesis_v2 import config, figuras, io_datos
from tesis_v2.io_datos import llave, fecha

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170, "display.max_columns", 30, "display.max_rows", 60)
figuras.estilo()

ds = io_datos.leer_parquet(config.PRIVADO / "dataset_analitico_v2.parquet")
lib = ds[(ds["producto"] == "libranza") & (ds["anio_cohorte"] != 2021)].copy()
lib["semestre"] = lib["t0"].dt.year.astype(str) + "-S" + np.where(lib["t0"].dt.month <= 6, "1", "2")
dic = pd.read_csv(config.RESULTADOS / "fase1" / "diccionario_v2.csv")
NUM = [v for v, t in zip(dic["variable"], dic["tipo"]) if t == "num"]
print(f"Créditos de libranza no anulados (sin la cohorte piloto 2021): {len(lib)} | grupos de cliente: {lib['grupo_cliente'].nunique()}")

Créditos de libranza no anulados (sin la cohorte piloto 2021): 1083 | grupos de cliente: 574


## 1. Evolución de la cartera

In [2]:
comp = lib.groupby("trimestre_t0").agg(
    creditos=("cid", "size"), pct_recaudo_directo=("recaudo", lambda s: 100 * (s != "empresarial").mean()),
    pct_recurrentes=("es_recurrente", lambda s: 100 * s.mean()),
    pct_refinanciacion=("es_refinanciacion", lambda s: 100 * s.mean()),
    monto_mediano_millones=("monto", lambda s: s.median() / 1e6), plazo_mediano=("plazo_meses", "median"),
    pct_con_buro=("tiene_buro", lambda s: 100 * s.mean())).round(1)
io_datos.guardar_resultado(comp.reset_index(), "fase2", "composicion_por_trimestre")
comp

,creditos,pct_recaudo_directo,pct_recurrentes,pct_refinanciacion,monto_mediano_millones,plazo_mediano,pct_con_buro
trimestre_t0,,,,,,,
2022Q1,5,100.0,0.0,0.0,5.3,19.0,0.0
2022Q2,9,100.0,0.0,0.0,2.0,11.0,0.0
2022Q3,16,100.0,12.5,0.0,2.8,19.0,0.0
2022Q4,22,100.0,4.5,0.0,2.0,14.0,0.0
2023Q1,44,100.0,4.5,0.0,3.2,15.0,0.0
2023Q2,52,100.0,13.5,0.0,2.9,13.0,0.0
2023Q3,52,96.2,65.4,0.0,3.6,15.0,0.0
2023Q4,60,23.3,70.0,0.0,3.7,14.4,0.0
2024Q1,61,0.0,63.9,0.0,5.0,14.9,0.0


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1.5, 1]})
tab = pd.crosstab(lib["trimestre_t0"], lib["recaudo"])[["empresarial", "mensual", "quincenal"]]
abajo = np.zeros(len(tab))
for col, color in zip(tab.columns, [figuras.AZUL, figuras.NARANJA, figuras.VERDE]):
    axes[0].bar(range(len(tab)), tab[col], bottom=abajo, color=color, width=0.8, label=col)
    abajo += tab[col].values
axes[0].set_xticks(range(len(tab)), tab.index, rotation=70, fontsize=7)
axes[0].set_ylabel("Créditos originados")
axes[0].set_title("Originación por trimestre y modalidad de recaudo")
axes[0].legend(title="Recaudo", fontsize=7.5)
axes[0].grid(axis="x", visible=False)
x = range(len(comp))
axes[1].plot(x, comp["pct_recurrentes"], color=figuras.AZUL, marker="o", ms=3, label="Clientes recurrentes")
axes[1].plot(x, comp["pct_refinanciacion"], color=figuras.NARANJA, marker="s", ms=3, label="Refinanciaciones")
axes[1].plot(x, comp["pct_con_buro"], color=figuras.VERDE, marker="^", ms=3, label="Con consulta de buró")
axes[1].set_xticks(list(x)[::2], comp.index[::2], rotation=70, fontsize=7)
axes[1].set_ylabel("% de los créditos del trimestre")
axes[1].set_title("Cambios en la composición")
axes[1].legend(fontsize=7.5)
fig.tight_layout()
figuras.guardar(fig, "fase2", "E01", "evolucion_cartera",
                "La cartera crece con fuerza desde 2025, se vuelve más recurrente y el buró solo aparece en los últimos trimestres.")

**Lectura.** Tres hechos saltan a la vista y los tres importan para el modelo:

1. **La modalidad de recaudo es, en la práctica, un indicador de época.** Hasta 2023Q3 todos los créditos de libranza figuran con recaudo directo (mensual o quincenal), y desde 2024Q1 todos con recaudo empresarial. No hay ningún trimestre en que convivan de forma estable. Lo más probable es un cambio en la forma de registrar el campo, o en la operación del convenio, y no una característica del cliente. Por eso, la "fuerte señal" del recaudo mensual que aparecía en la v1 no se puede separar del efecto cohorte (2022–2023). **Es una pregunta para la operación** y, mientras se resuelve, `recaudo` no debe entrar al modelo como si fuera un rasgo del cliente.
2. **Las refinanciaciones solo aparecen desde 2024Q2** (la etiqueta se empezó a registrar entonces) y se vuelven frecuentes en 2025–2026. Los clientes recurrentes pasan de casi cero a dos tercios a mediados de 2023.
3. **El buró aparece solo desde 2026Q1.** La población de 2026, que es la que se califica hoy, no se parece a la de 2022–2024, con la que se aprende. La deriva se cuantifica en el notebook 07.

## 2. Selectividad de la aprobación (sesgo de selección)

Solo observo el desempeño de los créditos aprobados. Si la fintech rechaza a muchos solicitantes, el modelo aprende sobre una población ya filtrada. `bd_solicitud` existe desde mayo de 2023 y registra el resultado de cada solicitud.

In [4]:
sol = io_datos.leer_tabla("solicitud", ["id", "fechasolicitud", "estado", "tipocredito_id", "tipo_solicitud"])
sol = sol[llave(sol["tipocredito_id"]).isin([1, 2])]
sol["anio"] = fecha(sol["fechasolicitud"]).dt.year
apr = pd.crosstab(sol["anio"], sol["estado"], margins=True, margins_name="Total")
apr["%_rechazada"] = (100 * apr.get("RECHAZADA", 0) / apr["Total"]).round(1)
apr["%_rechazada_sobre_decididas"] = (100 * apr.get("RECHAZADA", 0) / (apr.get("RECHAZADA", 0) + apr.get("APROBADA", 0))).round(1)
io_datos.guardar_resultado(apr.reset_index(), "fase2", "aprobacion_solicitudes_libranza")
apr

estado,APROBADA,EN_ESTUDIO,RECHAZADA,Total,%_rechazada,%_rechazada_sobre_decididas
anio,,,,,,
2023,103,9,6,118,5.1,5.5
2024,280,11,47,338,13.9,14.4
2025,226,15,87,328,26.5,27.8
2026,326,37,89,452,19.7,21.4
Total,935,72,229,1236,18.5,19.7


**Lectura.** En total, una de cada cinco solicitudes de libranza decididas se rechaza (19,7 %), pero la tasa cambió mucho: 5,5 % en 2023, 14 % en 2024 y 28 % en 2025. La política se endureció con el tiempo, y eso también es deriva: los aprobados de 2025 pasaron un filtro más estricto que los de 2023. El sesgo de selección existe y no es despreciable: el modelo describe el riesgo **entre aprobados**, y su uso para decidir sobre todos los solicitantes extrapola. No hay desempeño de los rechazados para corregirlo, así que lo declaro como limitación. En la Fase 4 haré un análisis de sensibilidad simple (por ejemplo, asignar a los rechazados una tasa de evento k veces la de los aprobados).

## 3. Concentración por empleador

En libranza el pagador es el empleador, que descuenta la cuota de la nómina. Si unos pocos empleadores concentran la cartera, el riesgo de un empleador (quiebra, despidos, retraso en giros) se transmite a muchos créditos a la vez. Reporto la concentración sin nombres.

In [5]:
emp = io_datos.leer_tabla("credito", ["id", "empresa_id"]).assign(cid=lambda x: llave(x["id"]), emp=lambda x: llave(x["empresa_id"]))
lib_e = lib.merge(emp[["cid", "emp"]], on="cid", how="left")
cuota = lib_e["emp"].value_counts(normalize=True)
conc = pd.DataFrame([{
    "empleadores_distintos": int(lib_e["emp"].nunique()), "creditos_sin_empleador": int(lib_e["emp"].isna().sum()),
    "participacion_top1_%": round(100 * cuota.iloc[0], 1), "participacion_top5_%": round(100 * cuota.iloc[:5].sum(), 1),
    "participacion_top10_%": round(100 * cuota.iloc[:10].sum(), 1), "HHI": round(10000 * (cuota ** 2).sum(), 0)}])
io_datos.guardar_resultado(conc, "fase2", "concentracion_empleadores")
conc

,empleadores_distintos,creditos_sin_empleador,participacion_top1_%,participacion_top5_%,participacion_top10_%,HHI
0,36,0,41.8,78.5,91.4,2211.0


**Lectura.** La cartera está concentrada: el principal empleador reúne el 42 % de los créditos, los cinco primeros el 79 % y los diez primeros el 91 %. El índice de Herfindahl-Hirschman (2.211) está en la franja alta de la concentración moderada. Hay dos implicaciones metodológicas. La primera es que el empleador induce dependencia entre créditos, además de la del cliente. En la Fase 4 voy a reportar una validación agrupada por empleador como sensibilidad. La segunda es que una variable "empleador" con muchos niveles memorizaría a los grandes convenios. Por eso solo entra su antigüedad (`emp_creditos_previos`).

## 4. Distribuciones univariadas

In [6]:
def resumen(x):
    x = x.astype(float)
    xs = x.dropna()
    return pd.Series({"n": len(xs), "%_faltante": 100 * x.isna().mean(), "media": xs.mean(), "desv": xs.std(),
                      "p01": xs.quantile(.01), "p25": xs.quantile(.25), "mediana": xs.median(), "p75": xs.quantile(.75),
                      "p99": xs.quantile(.99), "asimetria": stats.skew(xs) if len(xs) > 2 else np.nan,
                      "%_ceros": 100 * (xs == 0).mean()})


uni = lib[NUM].apply(resumen).T.round(2)
io_datos.guardar_resultado(uni.reset_index(names="variable"), "fase2", "univariado_numericas")
uni

,n,%_faltante,media,desv,p01,p25,mediana,p75,p99,asimetria,%_ceros
monto,1083.0,0.00,5611825.24,9121573.65,400000.00,1474500.00,2669100.00,5915187.00,4.101600e+07,5.97,0.00
n_cuotas,1083.0,0.00,18.00,14.16,1.00,10.00,13.00,23.00,5.500000e+01,6.09,0.00
plazo_meses,1083.0,0.00,18.11,14.17,1.02,10.04,13.44,23.59,5.478000e+01,6.08,0.00
cuota_mensual,1082.0,0.09,448914.04,1277431.65,63616.66,151337.49,238838.71,402260.20,2.612504e+06,11.54,0.00
tasa_mensual,1083.0,0.00,1.81,0.35,1.35,1.57,1.79,1.98,2.790000e+00,0.82,0.00
aval_pct,637.0,41.18,4.21,1.57,2.00,2.00,5.00,5.00,6.000000e+00,-0.28,0.00
monto_sobre_solicitado,145.0,86.61,2.30,1.93,1.07,1.31,1.69,2.45,1.020000e+01,4.06,0.00
edad_t0,1080.0,0.28,38.82,10.99,20.25,30.49,36.94,46.62,6.294000e+01,0.47,0.00
n_creditos_previos,1083.0,0.00,1.01,1.45,0.00,0.00,0.00,2.00,6.000000e+00,1.80,52.82
n_activos_t0,1083.0,0.00,0.35,0.51,0.00,0.00,0.00,1.00,2.000000e+00,1.09,66.67


In [7]:
VARS_HIST = ["monto", "plazo_meses", "cuota_mensual", "tasa_mensual", "edad_t0", "n_creditos_previos",
             "peor_mora_previa", "meses_desde_primer_credito", "emp_creditos_previos"]
LOG = {"monto", "cuota_mensual", "peor_mora_previa", "emp_creditos_previos"}
fig, axes = plt.subplots(3, 3, figsize=(10, 7.2))
for ax, v in zip(axes.flat, VARS_HIST):
    x = lib[v].dropna().astype(float)
    if v in LOG:
        x = np.log10(x + 1)
    ax.hist(x, bins=35, color=figuras.AZUL, edgecolor="white", linewidth=0.4)
    ax.axvline(x.median(), color=figuras.NARANJA, lw=1.4)
    ax.set_title(v + (" (log10(x+1))" if v in LOG else ""), fontsize=8.5)
    ax.set_ylabel("Créditos", fontsize=7.5)
fig.suptitle("Distribución de las variables numéricas — libranza (línea = mediana)", fontweight="bold", fontsize=10)
fig.tight_layout()
figuras.guardar(fig, "fase2", "E02", "histogramas_numericas",
                "Montos, cuotas y mora previa tienen colas largas a la derecha; la edad y la tasa son casi simétricas.")

**Lectura.** Las variables monetarias y la mora previa son muy asimétricas (asimetría entre 6 y 11). Es normal en crédito y conviene trabajarlas en logaritmo o con intervalos. La tasa mensual está muy concentrada alrededor de 1,8 %: casi no varía entre clientes y por eso difícilmente discrimina. La peor mora previa tiene una masa grande en cero y una cola de pocos clientes con moras largas.

In [8]:
CATS = ["recaudo", "tipo_solicitud", "cierre_previo", "tiene_aval", "cobra_seguro", "cobra_estudio", "cobra_4x1000",
        "es_recurrente", "sin_empleador"]
fig, axes = plt.subplots(3, 3, figsize=(10, 6.6))
frec = []
for ax, v in zip(axes.flat, CATS):
    vc = lib[v].astype(str).value_counts()
    frec.append(pd.DataFrame({"variable": v, "categoria": vc.index, "creditos": vc.values,
                              "%": (100 * vc.values / vc.sum()).round(1)}))
    ax.barh(vc.index[::-1], vc.values[::-1], color=figuras.AZUL, height=0.65)
    for i, (n_, p_) in enumerate(zip(vc.values[::-1], 100 * vc.values[::-1] / vc.sum())):
        ax.text(n_, i, f" {p_:.0f} %", va="center", fontsize=7)
    ax.set_title(v, fontsize=8.5)
    ax.set_xlim(0, vc.max() * 1.25)
    ax.grid(axis="y", visible=False)
fig.suptitle("Frecuencia de las variables categóricas — libranza", fontweight="bold", fontsize=10)
fig.tight_layout()
io_datos.guardar_resultado(pd.concat(frec), "fase2", "frecuencias_categoricas")
figuras.guardar(fig, "fase2", "E03", "frecuencias_categoricas",
                "El 80 % de los créditos figura con recaudo empresarial y el 53 % son de clientes sin crédito previo.")

## 5. Faltantes: cuándo y por qué

Calculo el porcentaje de faltantes por variable y trimestre. Para clasificarlos uso una regla simple. Un faltante es **estructural** si la variable no existe (≥ 95 % de faltantes) en los trimestres anteriores a la entrada de la fuente y luego aparece. Es **por diseño** si depende de una condición del crédito (por ejemplo, la mora previa solo existe si hay un crédito previo). Es **ocasional** si es poco frecuente y sin patrón temporal claro (prueba χ² de faltante frente a trimestre).

In [9]:
miss = lib[NUM].isna().groupby(lib["trimestre_t0"]).mean().mul(100).T
miss = miss[miss.max(axis=1) > 0]
fig, ax = plt.subplots(figsize=(10, 0.28 * len(miss) + 1.6))
im = ax.imshow(miss.values, aspect="auto", cmap="Blues", vmin=0, vmax=100)
ax.set_xticks(range(miss.shape[1]), miss.columns, rotation=70, fontsize=7)
ax.set_yticks(range(miss.shape[0]), miss.index, fontsize=7.5)
ax.grid(False)
cb = fig.colorbar(im, ax=ax, fraction=0.025)
cb.set_label("% faltante", fontsize=8)
ax.set_title("Porcentaje de faltantes por variable y trimestre de originación")
figuras.guardar(fig, "fase2", "E04", "mapa_faltantes",
                "El buró y la PILA faltan por completo antes de dic-2025: es un faltante estructural, no aleatorio.")

In [10]:
filas = []
for v in miss.index:
    m = lib[v].isna()
    tab = pd.crosstab(lib["trimestre_t0"], m)
    p = stats.chi2_contingency(tab)[1] if tab.shape[1] == 2 else np.nan
    por_trim = m.groupby(lib["trimestre_t0"]).mean()
    primer = por_trim[por_trim < 0.95].index.min()
    previos = por_trim[por_trim.index < primer] if isinstance(primer, str) else pd.Series(dtype=float)
    if v in {"peor_mora_previa", "meses_desde_primer_credito"}:
        tipo = "Por diseño (solo si hay crédito previo)"
    elif v == "aval_pct":
        tipo = "Por diseño (solo si hay aval)"
    elif len(previos) >= 2 and (previos >= 0.95).all():
        tipo = f"Estructural (la fuente existe desde {primer})"
    elif m.mean() < 0.05:
        tipo = "Ocasional"
    else:
        tipo = "Mixto"
    filas.append({"variable": v, "%_faltante_total": round(100 * m.mean(), 1), "primer_trimestre_con_dato": primer,
                  "p_chi2_vs_trimestre": p, "clasificacion": tipo})
clas = pd.DataFrame(filas).sort_values("%_faltante_total", ascending=False)
io_datos.guardar_resultado(clas, "fase2", "clasificacion_faltantes")
clas

,variable,%_faltante_total,primer_trimestre_con_dato,p_chi2_vs_trimestre,clasificacion
2,monto_sobre_solicitado,86.6,2024Q2,3.240619e-43,Estructural (la fuente existe desde 2024Q2)
19,pila_salario_sobre_declarado,85.1,2026Q2,7.808732e-112,Estructural (la fuente existe desde 2026Q2)
18,pila_antiguedad_aportante_meses,85.0,2026Q2,7.002723e-113,Estructural (la fuente existe desde 2026Q2)
17,pila_meses_cotizados_12m,85.0,2026Q2,7.002723e-113,Estructural (la fuente existe desde 2026Q2)
16,carga_financiera_buro,82.3,2026Q1,2.756349e-105,Estructural (la fuente existe desde 2026Q1)
13,cuota_total_buro,82.3,2026Q1,2.756349e-105,Estructural (la fuente existe desde 2026Q1)
12,valor_en_mora_buro,82.2,2026Q1,3.555266e-106,Estructural (la fuente existe desde 2026Q1)
9,veces_30mas_12m,81.3,2026Q1,4.186479e-113,Estructural (la fuente existe desde 2026Q1)
10,veces_90mas_12m,81.3,2026Q1,4.186479e-113,Estructural (la fuente existe desde 2026Q1)
8,aperturas_6m,81.1,2026Q1,1.023250e-115,Estructural (la fuente existe desde 2026Q1)


**Lectura.** Casi todo el faltante es estructural o por diseño, no aleatorio. Esto cambia la estrategia de imputación de la v1:

- El buró y la PILA no se pueden imputar en 2022–2024 con supuestos de MAR. No existían, y "imputarlas" con donantes de Home Credit es inventar una relación que nunca se observó. En el modelo principal quedan fuera y se usan solo en el análisis que incluye créditos recientes (supervivencia) o vía *prior*.
- La mora previa y la antigüedad del cliente faltan porque no hay crédito previo. Ese faltante es informativo: se codifica con `es_recurrente` y un valor neutro, no con imputación.
- Lo ocasional (edad en 3 casos, una cuota) se imputa con la mediana del pliegue sin pérdida.

Hay una advertencia sobre el buró: `veces_30mas_12m` y `veces_90mas_12m` tienen medianas de 12 y valores de hasta 90. Eso no es compatible con un "número de veces en mora en 12 meses". Los atributos de CreditVision que la v1 mapeó con esos nombres (`G001B`, `G003S`) probablemente miden otra cosa, por ejemplo meses desde la última mora. Su semántica se debe verificar con el diccionario de TransUnion antes de usarlos.

## 6. Redundancia entre variables

In [11]:
cand = [v for v in NUM if lib[v].notna().mean() > 0.4]
rho = lib[cand].astype(float).corr(method="spearman", min_periods=100)
fig, ax = plt.subplots(figsize=(7.5, 6.2))
im = ax.imshow(rho.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cand)), cand, rotation=70, ha="right", fontsize=7.5)
ax.set_yticks(range(len(cand)), cand, fontsize=7.5)
for i in range(len(cand)):
    for j in range(len(cand)):
        if np.isfinite(rho.values[i, j]) and i != j:
            ax.text(j, i, f"{rho.values[i, j]:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if abs(rho.values[i, j]) > 0.6 else "black")
ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.04)
ax.set_title("Correlación de Spearman entre predictores numéricos — libranza")
figuras.guardar(fig, "fase2", "E05", "correlacion_spearman",
                "Plazo y número de cuotas son casi la misma variable, y monto y cuota están fuertemente ligados: hay que elegir uno de cada par.")
pares = rho.where(np.triu(np.ones(rho.shape, bool), 1)).stack().rename("rho").reset_index()
pares = pares[pares["rho"].abs() >= 0.7].sort_values("rho", key=abs, ascending=False)
io_datos.guardar_resultado(pares.rename(columns={"level_0": "variable_1", "level_1": "variable_2"}), "fase2", "pares_redundantes")
pares

,level_0,level_1,rho
14,n_cuotas,plazo_meses,0.998433
3,monto,cuota_mensual,0.798424
92,n_creditos_previos,n_activos_t0,0.753575


In [12]:
_ = io_datos.escribir_registro("05_eda_descriptivo_y_faltantes", {"creditos": len(lib), "figuras": ["E01", "E02", "E03", "E04", "E05"]})

## Conclusiones

- La cartera de libranza creció con fuerza desde 2025, se volvió más recurrente y cambió de composición. La población de aprendizaje (2022–2024) no es la misma que la población que se califica hoy.
- **La modalidad de recaudo coincide casi uno a uno con la época** (directo hasta 2023Q3, empresarial desde 2024). No puede leerse como rasgo del cliente. Es una pregunta para Eider y la operación.
- El rechazo de solicitudes pasó de 5,5 % (2023) a 28 % (2025): el modelo describe solo a los aprobados, bajo una política que se endureció.
- La cartera está concentrada en pocos empleadores (42 % en el primero, HHI 2.211). Esto añade una segunda fuente de dependencia que se revisará como sensibilidad.
- El faltante es casi todo estructural o por diseño. El buró no se imputa en cohortes donde no existía, y la ausencia de historia se codifica, no se imputa.
- Pares redundantes: plazo ≈ número de cuotas, monto ≈ cuota mensual. Para la Fase 3 propongo conservar el plazo y el monto.